In [1]:
1+1

2

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
from langsmith import Client 
client = Client()

dataset_name = "Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['6319b7df-d760-4c63-aad1-85e2dd4f8472',
  'e0da9789-7113-4df2-9d75-7f2431b462ce',
  '42674516-b5b8-4ca3-b790-906374ae479f',
  '3d7a43e1-664f-4648-a70b-da3806b84b28',
  'a0a2fcce-7583-414e-ab5c-d92269b199b9'],
 'count': 5,
 'as_of': '2026-07-21T05:49:09.688462635Z'}

In [56]:
from google import genai
from google.genai import types
from langsmith import wrappers
import os

gemini_client = wrappers.wrap_gemini(
    genai.Client(api_key=os.environ["GEMINI_API_KEY"])
)

eval_instructions = (
    "You are an expert professor specialized in grading students' answers. "
    "Respond with ONLY one word: CORRECT or INCORRECT."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict):
    user_content = f"""
Question:
{inputs['question']}

Reference Answer:
{reference_outputs['answer']}

Student Answer:
{outputs['response']}

Is the student answer correct?

Respond with ONLY:
CORRECT
or
INCORRECT
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_content,
        config=types.GenerateContentConfig(
            system_instruction=eval_instructions,
            temperature=0,
        ),
    )

    grade = response.text.strip().upper()

    return {
        "key": "correctness",
        "score": 1 if grade == "CORRECT" else 0,
    }

In [57]:
def concision(inputs: dict, outputs: dict, reference_outputs: dict):
    return {
        "key": "concision",
        "score": int(
            len(outputs["response"]) < 2 * len(reference_outputs["answer"])
        ),
    }

In [58]:
from google.genai import types

default_instructions = (
    "Respond to the user's question in a short, concise manner (one short sentence)."
)

def my_app(
    question: str,
    model: str = "gemini-3.5-flash",
    instructions: str = default_instructions,
) -> str:

    response = gemini_client.models.generate_content(
        model=model,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=0,
        ),
    )

    return response.text

In [59]:
#call my_app for every datapoints
def ls_target(inputs:str) -> dict:
    return {"response":my_app(inputs["question"])}

In [60]:
experiments_result = client.evaluate(
    ls_target,
    data = dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="genai-3.5-flash"
)

View the evaluation results for experiment: 'genai-3.5-flash-c7ad8654' at:
https://smith.langchain.com/o/45409f9a-a636-444b-828f-e8275b159680/datasets/aeff6aa6-ac20-4f91-83b6-38e737d89394/compare?selectedSessions=681c417b-c162-493b-aaf9-6b27ea66ea7b




3it [00:10,  3.61s/it]Error running target function: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 6.735383063s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': '

1

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")

In [3]:
from langsmith import Client

client = Client()

dataset_name = "LLM Evaluation begineer "
examples = [
    {
        "inputs": {
            "question": "What is machine learning?"
        },
        "outputs": {
            "answer": "Machine learning is a branch of AI that allows computers to learn from data."
        }
    },

    {
        "inputs": {
            "question": "What is Python?"
        },
        "outputs": {
            "answer": "Python is a high-level programming language."
        }
    },

    {
        "inputs": {
            "question": "What is SQL?"
        },
        "outputs": {
            "answer": "SQL is a language used to manage and query relational databases."
        }
    },

    {
        "inputs": {
            "question": "What is deep learning?"
        },
        "outputs": {
            "answer": "Deep learning is a type of machine learning based on neural networks with multiple layers."
        }
    },

    {
        "inputs": {
            "question": "What is LangSmith?"
        },
        "outputs": {
            "answer": "LangSmith is a platform for tracing, evaluating, testing, and monitoring LLM applications."
        }
    }
]

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Begineer dataset for evaluating an llm application"
)

client.create_examples(
    inputs = [example["inputs"] for example in examples],
    outputs = [example["outputs"] for example in examples],
    dataset_id=dataset.id,
)

print("Dataset Created Sucessfully")
print(f"Dataset {dataset.name}")
print(f"Dataset id{dataset.id}")

Dataset Created Sucessfully
Dataset LLM Evaluation begineer
Dataset ide395546c-f821-4349-ba64-a698d9f05517


In [4]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature= 0
)
model.invoke("What is Machine Learning")

c:\Basicrag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content="**Machine Learning (ML)** is a subset of Artificial Intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to automatically improve their performance on a task by learning from experience, rather than relying on human intervention.\n\n**Key Characteristics of Machine Learning:**\n\n1. **Data-driven**: ML algorithms learn from data, which can be in the form of images, text, audio, or other types of input.\n2. **Pattern recognition**: ML algorithms identify patterns in the data to make predictions or decisions.\n3. **Self-improvement**: ML algorithms can improve their performance over time as they receive more data and learn from their mistakes.\n4. **Autonomy**: ML algorithms can operate independently, without human intervention, once they have been trained.\n\n**Types of Machine Learning:**\n\n1. **Supervised Learning**: The algorithm is trained on labeled data

In [5]:
def target(inputs:dict)->dict:
    question = inputs["question"]
    
    prompt = f"""Answer the following question accuarately and concisely
    Question:
    {question}
    """
    
    response = model.invoke(prompt)
    
    return {
        "answer":response.content
    }

In [6]:
result = target({
    "question":"what is machine learning"
})

print(result)

{'answer': '**Machine Learning (ML)**: A subset of artificial intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables systems to improve their performance on a task over time, based on experience and data.'}


In [7]:
#LLM AS A JUDGE

judge_llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature=0
)

In [15]:
#Creating a correctness evaluator
def correctness_evaluator(
    inputs:dict,
    outputs:dict,
    reference_outputs:dict
):
    question = inputs["question"]
    answer = outputs["answer"]
    expected = reference_outputs["answer"]
    
    prompt = f"""
    You are an expert evaluator
    
    
    Evaluate Whether the AI generated answer is corrcet or not
    
    Question:{question},
    
    answer:{answer},
    
    expected_answer:{expected}
    
    Give a score:

1 = Completely incorrect
2 = Mostly incorrect
3 = Partially correct
4 = Mostly correct
5 = Completely correct

Return ONLY the number.
    """
    
    response = judge_llm.invoke(prompt)
    
    try:
        score_5 = int(response.content.strip())
        
        # Convert 1-5 score to 0-1
        score = score_5 / 5.0

    except ValueError:
        score = 0.0
        
    return {
        "key":"correctness",
        "score":score
    }
      

In [16]:
results = client.evaluate(
    target,
    data = dataset,
    evaluators=[
        correctness_evaluator
    ],
    experiment_prefix="llama-3.3-70b-evaluator",
    max_concurrency=2
)
print("Evaluation Completed")

View the evaluation results for experiment: 'llama-3.3-70b-evaluator-9fb03416' at:
https://smith.langchain.com/o/45409f9a-a636-444b-828f-e8275b159680/datasets/e395546c-f821-4349-ba64-a698d9f05517/compare?selectedSessions=8580d32d-3785-49c0-a6ea-5eed4c78b733




5it [00:01,  3.26it/s]

Evaluation Completed


In [17]:
def exact_match(inputs,outputs,reference_outputs):
    
    predicted = outputs["answer"].strip().lower()
    expected = reference_outputs["answer"].strip().lower()
    
    return {
        "key":"exact_match",
        "score":int(predicted == expected)
    }

In [19]:
results = client.evaluate(
    target,
    data = dataset,
    evaluators=[
        exact_match
    ],
    experiment_prefix="llama-3.3-70b-versatile",
    max_concurrency=2
)
print(results)

View the evaluation results for experiment: 'llama-3.3-70b-versatile-ec6c1d2f' at:
https://smith.langchain.com/o/45409f9a-a636-444b-828f-e8275b159680/datasets/e395546c-f821-4349-ba64-a698d9f05517/compare?selectedSessions=9318d746-46fb-4a26-8d90-a29decebb818




5it [00:02,  2.44it/s]

<ExperimentResults llama-3.3-70b-versatile-ec6c1d2f>


In [20]:
def relavance(inputs,outputs,reference_outputs):
    question = inputs["question"]
    answer = outputs["answer"]
    expected = reference_outputs["answer"]
    
    prompt = f"""
    You are an Experience Evaluator ,
    Evaluate the answer provoided by AI where the answer is relavance to question {
        question
    }:
    expected_answer:
    {expected}:
    ai_answer:
    {answer}
    
    Evaluate whether the AI answer is correct.

Score:

1 = Completely incorrect
2 = Mostly incorrect
3 = Partially correct
4 = Mostly correct
5 = Completely correct

Return ONLY the number.
    """
    response = judge_llm.invoke(prompt)
    try:
        score = int(response.content.strip())
        
        score = score/5.0
        
    except ValueError:
        score = 0.0
        
    return {
        "key":"relavence",
        "score":score
    }
        
    

In [24]:
results = client.evaluate(
    target,
    data = dataset,
    evaluators=[relavance],
    max_concurrency=2,
    experiment_prefix="llama-3.3-relavance"
)
print(results)

View the evaluation results for experiment: 'llama-3.3-relavance-e0f127b2' at:
https://smith.langchain.com/o/45409f9a-a636-444b-828f-e8275b159680/datasets/e395546c-f821-4349-ba64-a698d9f05517/compare?selectedSessions=dd2c4b2b-a77b-4249-aad4-12870a0090a4




5it [00:01,  4.05it/s]

<ExperimentResults llama-3.3-relavance-e0f127b2>


In [1]:
##Evaluation for RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#initialize a text splitter 
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 250,chunk_overlap = 0)

doc_splitter = text_splitter.split_documents(docs_list)

#Add documents to the vector store using opeai embeddings
vector_store = InMemoryVectorStore.from_documents(
    documents=doc_splitter,
    embedding=HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2"
    )
)

retriver = vector_store.as_retriever(k = 6)

C:\Users\KEERTHI VARDHAN\AppData\Local\Temp\ipykernel_25376\665672957.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Basicrag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1696.09it/s]


In [2]:
retriver.invoke("what is agents")

[Document(id='5981f719-7ece-4f08-885d-d6c9ccc2b3ca', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [29]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model = "qwen/qwen3.6-27b",
    model_provider="groq"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.9'}}, client=<groq.resources.chat.completions.Completions object at 0x0000028F51BA2A80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028F51BA3200>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [30]:
from langsmith import traceable

@traceable
def rag_bot(question:str)->dict:
    docs = retriver.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)
    
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    ai_msg = llm.invoke(
    [
        {"role":"system","content":instructions},
        {"role":"user","content":question}
    ]
)
    return {"answer":ai_msg.content,"documents":docs}

In [11]:
rag_bot("what is agents")

{'answer': '\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Question: "what is agents"\n   - Context: The user is asking about "agents" based on the provided source documents.\n   - Constraint: Use three sentences maximum, keep it concise, rely only on the provided text.\n\n2.  **Analyze Source Documents:**\n   - "This benchmark evaluates the agent’s tool use capabilities at three levels: Generative Agents (Park, et al. 2023) is super fun experiment where 25 virtual characters, each controlled by a LLM-powered agent, are living and interacting in a sandbox environment, inspired by The Sims."\n   - "Generative agents create believable The generative agent architecture. (Image source: Park et al. 2023) Relationships between agents and observations of one agent by another are all taken into consideration for planning and reacting."\n   - "Environment information is present in a tree structure."\n\n3.  **Extract Key Information about "Agents":**\n   - Agents are

In [12]:
from langsmith import Client
client = Client()

examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

dataset_name = "Rag Evaluation"
if client.has_dataset(dataset_name=dataset_name):
    print("Dataset already exists.using existing dataset")
    
    dataset = client.read_dataset(dataset_name=dataset_name)
else:
    print("Dataset Does not exist creating it")
    
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Beginner dataset for LLM evaluation"
    )
    client.create_examples(
    examples=examples,
    dataset_id=dataset.id
    )
    print("Dataset created Sucessfully")
print(f"Dataset name{dataset_name}")
print(f"Datasetid{dataset.id}")

Dataset already exists.using existing dataset
Dataset nameRag Evaluation
Datasetida92740b8-5c84-4a2d-afc1-877b15b47403


In [37]:
##Evaluators

from typing_extensions import Annotated,TypedDict

class CorrectnessGrade(TypedDict):
    explanation:Annotated[str,...,"Explain your reasoning for the score"]
    correct:Annotated[bool,...,"True if the answer is correct,False Otherwise"]
    
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq
graded_llm = ChatGroq(
    model = "qwen/qwen3.6-27b",
    temperature=0
).with_structured_output(CorrectnessGrade,method="json_schema",strict=True)

def correctness(inputs:dict,outputs:dict,reference_outputs:dict)->bool:
    """ An evaluator for RAG answer acuuracy"""
    answers = f"""
    Question:{inputs["question"]},
    Ground_Truth_answer:{reference_outputs["answer"]},
    Student_answer:{outputs["answer"]}
    """
    
    grade = graded_llm.invoke([
        {"role":"system","content":correctness_instructions},
        {"role":"user","content":answers}
    ])
    return grade["correct"]

In [24]:
class RelavanceGrade(TypedDict):
    explanation:Annotated[str,...,"Explain your reasoning for the score"]
    relevant:Annotated[bool,...,"Provide  me the score whether the answer adresses the question"]
    
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

graded_llm = ChatGroq(
    model = "qwen/qwen3.6-27b",
    temperature=0
).with_structured_output(RelavanceGrade,method="json_schema",strict=True)

def relavence(inputs:dict,outputs:dict)->bool:
    """A simple evaluator for RAG relavance answer for the question"""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    
    grade = graded_llm.invoke([
        {"role":"system","content":relevance_instructions},
        {"role":"user","content":answer}
    ])
    return grade["relevant"]

Grooundness

In [38]:
class GroundnessGrade(TypedDict):
    explanation:Annotated[str,...,"Explain your reasoning for the score"]
    groundness:Annotated[bool,...,"Provide the score on if the answer hallunicates from the documents"]
    
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

graded_llm = ChatGroq(
    model = "qwen/qwen3.6-27b",
    temperature=0
).with_structured_output(GroundnessGrade,method="json_schema",strict=True)

def groundness(inputs:dict,outputs:dict)-> bool:
    """A simple Evaluator for RAG answer Groundness"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"Facts {doc_string}\n STUDENT_ANSWER :{outputs["answer"]}"
    grade = graded_llm.invoke([
        {"role":"system","content":grounded_instructions},
        {"role":"user","content":answer}
    ]
    )
    return grade["groundness"]

In [ ]:
# Retrived docs vs input
class RetrivedGrade(TypedDict):
    explanation:Annotated[str,...,"Explain the reasoning for the score"]
    relavances:Annotated[bool,...,"True if teh retrived documents are relavent to the question,otherwise false"]
    
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0
).with_structured_output(RetrivedGrade,method='json_schema',strict=True)

def retrival_relavance(inputs:dict,outputs:dict)->bool:
    """A Simple RAG Evaluator for the retrived relavance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"Facts;{doc_string}\nQuestion{inputs["question"]}"
    grade = llm.invoke([
        {"role":"system","content":retrieval_relevance_instructions},
        {"role":"user","content":answer}
    ])
    return grade["relavances"]

In [40]:
def target(inputs)->dict:
    result = rag_bot(inputs["question"])
    return {
        "answer": result["answer"]
    }

In [41]:
test = target({
    "question": "What is an AI agent?"
})

print(test)

{'answer': '\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Question: "What is an AI agent?"\n   - Constraint: Use provided source documents, answer in max 3 sentences, keep it concise. If unknown, say so.\n\n2.  **Analyze Source Documents:**\n   - Document 1: "The generative agent architecture. (Image source: Park et al. 2023) Given the user request and the call command, the AI assistant helps the user to select a suitable model from a list of models to process the user request. The AI assistant merely outputs the model id of the most appropriate model."\n   - Document 2: "The output must be Generative Agents (Park, et al. 2023) is super fun experiment where 25 virtual characters, each controlled by a LLM-powered agent, are living and interacting in a sandbox environment, inspired by The Sims."\n   - Document 3: "Generative agents create believable Each element is an observation, an event directly provided by the agent."\n   - Document 4: "- Inter-agent com

In [ ]:

response = client.evaluate(
    target,
    data= dataset,
    evaluators=[
        CorrectnessGrade,
        RelavanceGrade,
        GroundnessGrade,
        RetrivedGrade
    ],
    experiment_prefix="rag-evaluation",
)